In [ ]:
# Formatting notebook to fit the browser size

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
# Loading libraries and packages

import os
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mtick
import scipy.optimize
import scipy.stats
import scipy.integrate as integrate
import scipy.integrate as special
import pandas as pd
import linecache
from statistics import mean
import matplotlib.font_manager as font_manager
plt.style.use('ggplot') # use ggplot style for graps

In [ ]:
#Taking optional values which are fed into the Running script
#If not present in Running script they are set to default values

if 'x_value_to_plot_to' in globals():
      max_x_value = x_value_to_plot_to
else:
    max_x_value = 50
    
    
if 'display_initial_parameter_graphs' in globals():
    display_initial_parameter_graphs = display_initial_parameter_graphs
    PATH = 'output/Graphs_of_initial_parameters'
    if not os.path.exists(PATH):
            os.makedirs(PATH)
else:
    display_initial_parameter_graphs = 'FALSE'


if 'normal_initial_A' in globals():
    A_2_ = normal_initial_A
else:
    A_2_ = 100

if 'normal_initial_mu' in globals():
    mu_2_ = normal_initial_mu
else:
    mu_2_ = 5

if 'normal_initial_sigma' in globals():
    sigma_2_ = normal_initial_sigma
else:
    sigma_2_ = 2



if 'lognormal_initial_A' in globals():
    A_1_ = lognormal_initial_A
else:
    A_1_ = 200

if 'lognormal_initial_mu' in globals():
    mu_1_ = lognormal_initial_mu
else:
    mu_1_ = 2

if 'lognormal_initial_sigma' in globals():
    sigma_1_ = lognormal_initial_sigma
else:
    sigma_1_ = 1



In [ ]:
# Import data

# path of input file(s) - note that the file(s) has to be in a folder called data_inputs
input_data_file = os.path.join('data_inputs', filename) #note - this uses a variable called filename as the input file, if you are only running one file you can change this to the name of your file in quotation marks 

#getting the base name of the file which will be used to name the output file in the same way
name = ((os.path.splitext(input_data_file))[0]).lstrip('data_inputs\\') #This takes the file name, splits it at the extension (e.g. csv), then takes the first element of this and removes the data_inputs\ folder location from it, this results in the file name

# define column headings for the dataframe
headings_list = ['Bin_number','Bin_diameter_lower', 'Diff_volume', 'Diff_number_perc', 'Diff_number_gram', 'Diff_surface_area', 'Diff_number']

# read in dataframe using pandas library 
input_data_df = pd.read_csv(input_data_file, sep='\,', engine='python', skiprows=55,
                            names=headings_list)

# calculate bin width in a new column
input_data_df['Bin_width'] = input_data_df['Bin_diameter_lower'].shift(-1) - input_data_df['Bin_diameter_lower']

# read off the final bin number into a variable, called input_data_bin_max, and then drop this row
input_data_bin_max = input_data_df.iloc[-1,1]
input_data_df = input_data_df.drop(input_data_df.index[input_data_df.tail(1).index])

# set 'Bin_number' column to be the index
input_data_df = input_data_df.set_index('Bin_number')

# change datatype of index from float to integer
input_data_df.index = input_data_df.index.astype('int')

# Calculate mid-point of bins
input_data_df['Bin_midpoint'] = input_data_df['Bin_diameter_lower']+input_data_df['Bin_width']/2

# Calc Diff_volume_density using: Diff_volume/Bin_width
input_data_df['Diff_volume_density'] = input_data_df['Diff_volume']/input_data_df['Bin_width']
Diff_volume_density = input_data_df['Diff_volume_density'].tolist()

# convert input data to lists
bin_midpoint_input_list = input_data_df['Bin_midpoint'].tolist()

In [ ]:
# NORMAL FUNCTION

In [ ]:
# Define the normal functions
def normal_func(x_, A_2_, mu_2_, sigma_2_):
    return (  (A_2_/(sigma_2_*np.sqrt(2.*np.pi)))*np.exp( -( (x_ - mu_2_)/sigma_2_ )**2/2. )  )
            

In [ ]:
# Define a function to fit the lognogrmal-normal function

#initial parameter values:

#default values
# A_2_ = 100
# mu_2_ = 5
# sigma_2_ = 2

initial_params_normal_ = [A_2_, mu_2_, sigma_2_]

def func_normal_fit(bin_midpoint_input_, Diff_volume_density):
    #initial parameter values:  
    fit_params_normal_, fit_cov_normal_ = scipy.optimize.curve_fit(normal_func,bin_midpoint_input_, Diff_volume_density, 
                                                    p0 = initial_params_normal_, 
                                                    bounds = ([0,0,0], [np.inf,np.inf,np.inf])) #note that the bounds of gamma have been fixed so that it can only take values from 0 to 1, everything else can take values from 0 to infinity
    fit_params_err_normal_ = np.sqrt(np.diag(fit_cov_normal_))
    return fit_params_normal_, fit_params_err_normal_

In [ ]:
# Run the normal fit 

#create empty lists which I will store failed/successful fittings in
failed_normal = []
successful_normal = []
fitting_failed_normal = []


#run the normal fit with the exception catcher
try:
    output_normal_fit_params_list, output_normal_fit_err = func_normal_fit(bin_midpoint_input_list, Diff_volume_density)

except RuntimeError:
    failed_normal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode

    #this sets the parameters to fitting failed so in the output file we get this message
    granule_diameter_normal = "fitting failed"
    granule_content_normal = "fitting failed"
    normal_standard_error_of_regression = "fitting failed"
    normal_uncertainity = "fitting failed"
    
    #this is used in an if statement during plotting so that if failed we don't get a graph
    fitting_failed_normal = "yes"

else:
    successful_normal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode
    
    #Output parameters - the fitting produces a dataframe called output_normal_fit_params_list which has the following format [A_2_, mu_2_, sigma_2_]
    A_normal = output_normal_fit_params_list[0]
    mu_normal = output_normal_fit_params_list[1]
    sigma_normal = output_normal_fit_params_list[2]
      
    #Getting the parameters from the models
    #norm mean is equal to mu
    granule_diameter_normal = mu_normal
    
    #normal variance is equal to sigma^2
    granule_variance_normal = sigma_normal**2
    
    #reduced uncertainity
    normal_uncertainity = np.sum(output_normal_fit_err)
    
    #uncertainty for each parameter - the fitting produces a dataframe of uncertainties called output_normal_fit_err which has the following format [A_2_, mu_2_, sigma_2_]
    A_normal_uncertainty = output_normal_fit_err[0]
    mu_normal_uncertainty = output_normal_fit_err[1]
    sigma_normal_uncertainty = output_normal_fit_err[2]
      
    
    #standard error of regression
    normal_vals_list = [normal_func(input_data_df['Bin_midpoint'].tolist()[i_], *output_normal_fit_params_list) #the star takes all the values from output_fit_params and puts it in here
                        for i_ in range(len(input_data_df['Diff_volume_density']))]
    normal_standard_error_of_regression = np.sqrt( (1/len(input_data_df['Diff_volume_density']-2)) * (sum((np.array(input_data_df['Diff_volume_density'].tolist()) - np.array(normal_vals_list))**2) / sum((np.array( np.array(input_data_df['Bin_midpoint'].tolist()) - mean(input_data_df['Bin_midpoint'].tolist()) ))**2)) )
    
    normal_variance = output_normal_fit_params_list[2]**2

In [ ]:
# LOGNORMAL FUNCTION

In [ ]:
# Define lognormal function

def lognormal_func(x_, A_1_, mu_1_, sigma_1_):
    return (  (A_1_/(x_*sigma_1_*np.sqrt(2.*np.pi)))*np.exp( -((np.log(x_)-mu_1_)/sigma_1_)**2/2. )  )

In [ ]:
# Define a function to fit the lognormal function

#initial parameter values:

#default values
A_1_ = 200
mu_1_ = 2
sigma_1_ = 1

initial_params_lognormal_ = [A_1_, mu_1_, sigma_1_]

def func_lognormal_fit(bin_midpoint_input_, Diff_volume_density):
    fit_params_lognormal_, fit_cov_lognormal_ = scipy.optimize.curve_fit(lognormal_func,bin_midpoint_input_, Diff_volume_density, 
                                                    p0 = initial_params_lognormal_, bounds = ([0,0,0], [np.inf,np.inf,np.inf]))
    fit_params_err_lognormal_ = np.sqrt(np.diag(fit_cov_lognormal_))
    return fit_params_lognormal_, fit_params_err_lognormal_

#This function will return: {list of fit parameters}, {list of fit uncertanties}

In [ ]:
# Run the lognormal fit 

#create empty lists which I will store failed/successful fittings in
failed_lognormal = []
successful_lognormal = []
fitting_failed_lognormal = []


#run the lognormal fit with the exception catcher
try:
    output_lognormal_fit_params_list, output_lognormal_fit_err = func_lognormal_fit(bin_midpoint_input_list, Diff_volume_density)

except RuntimeError:
    failed_lognormal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode

    #this sets the parameters to fitting failed so in the output file we get this message
    granule_diameter_lognormal = "fitting failed"
    granule_content_lognormal = "fitting failed"
    lognormal_standard_error_of_regression = "fitting failed"
    lognormal_uncertainity = "fitting failed"
    
    #this is used in an if statement during plotting so that if failed we don't get a graph
    fitting_failed_lognormal = "yes"

else:
    successful_lognormal = name.split('\n') #we need the \n if not we get \n included in the sample names when we make a list of samples in batch mode
    
    #Output parameters - the fitting produces a dataframe called output_lognormal_fit_params_list which has the following format [A_1_, mu_1_, sigma_1_]
    A_lognormal = output_lognormal_fit_params_list[0]
    mu_lognormal = output_lognormal_fit_params_list[1]
    sigma_lognormal = output_lognormal_fit_params_list[2]
      
    #Getting the parameters from the models
    #lognorm mean is equal to exp(mu + (sigma^2/2))
    granule_diameter_lognormal = np.exp(mu_lognormal + ((sigma_lognormal**2)/2))
    
    #lognorm variance is equal to [exp(sigma^2)-1] exp(2*mu + sigma^2)
    granule_variance_lognormal = ((np.exp(sigma_lognormal**2)-1)*(np.exp(2*mu_lognormal+sigma_lognormal**2)))
    
    #reduced uncertainity
    lognormal_uncertainity = np.sum(output_lognormal_fit_err)
    
    #uncertainty for each parameter - the fitting produces a dataframe of uncertainties called output_lognormal_fit_err which has the following format [A_1_, mu_1_, sigma_1_]
    A_lognormal_uncertainty = output_lognormal_fit_err[0]
    mu_lognormal_uncertainty = output_lognormal_fit_err[1]
    sigma_lognormal_uncertainty = output_lognormal_fit_err[2]
      
    #standard error of regression
    lognormal_vals_list = [lognormal_func(input_data_df['Bin_midpoint'].tolist()[i_], *output_lognormal_fit_params_list) #the star takes all the values from output_fit_params and puts it in here
                        for i_ in range(len(input_data_df['Diff_volume_density']))]
    lognormal_standard_error_of_regression = np.sqrt( (1/len(input_data_df['Diff_volume_density']-2)) * (sum((np.array(input_data_df['Diff_volume_density'].tolist()) - np.array(lognormal_vals_list))**2) / sum((np.array( np.array(input_data_df['Bin_midpoint'].tolist()) - mean(input_data_df['Bin_midpoint'].tolist()) ))**2)) )
    
    lognormal_variance = (np.exp(output_lognormal_fit_params_list[2]**2)-1)*(np.exp(2*output_lognormal_fit_params_list[1]+output_lognormal_fit_params_list[2]**2))

In [ ]:
# PLOTTING

In [ ]:
#Plotting initial parameter - optional extra    
#Plots the initial parameters to visualise how close the initial parameters fit
#This can be useful if your curves are very different from normal and the initial parameters need to be adjusted
#Will only run if display_initial_parameter_graphs = TRUE

if (display_initial_parameter_graphs == 'TRUE'):
    
    #setting up plots
    fig, ax = plt.subplots(2,1,figsize=(15,25))
    fig.suptitle(name, fontsize=20, y=1.01,   fontname="Arial", weight="bold")
    fig.subplots_adjust(top=0.95)
    fig.tight_layout(h_pad=8)
    font = font_manager.FontProperties(family='Arial',
                                       style='normal', size=11)
    axfont = 11
    
    # define x_values to plot
    x_vals_plot = np.arange(0.5,max_x_value,0.5)
    
    
    
    

    # define curves to fit   
    normal_vals_curve_initial = [normal_func(i_, initial_params_normal_[0], initial_params_normal_[1],initial_params_normal_[2]) 
                            for i_ in x_vals_plot]

    lognormal_vals_curve_initial = [lognormal_func(i_, initial_params_lognormal_[0],initial_params_lognormal_[1],initial_params_lognormal_[2]) 
                            for i_ in x_vals_plot]
    
    

    
        # define curves to fit
    for spine in ['left','right','top','bottom']:
        ax[0].spines[spine].set_color('k')
    ax[0].set_facecolor('white')
    ax[0].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
            linewidth=2, color='k',label='Data' )
    ax[0].plot(x_vals_plot, normal_vals_curve_initial,
            linewidth=2, color='b',label='Normal' )
    ax[0].plot(x_vals_plot, normal_vals_curve_initial,
            linewidth=2, color='w', alpha=0, label=('A = %.2f\nmu = %.2f\nsigma = %.2f ' ) % (initial_params_normal_[0], initial_params_normal_[1],initial_params_normal_[2])  )  
    ax[0].set_title('Initial parameters for Normal ', fontsize=16, fontname="Arial", weight="bold"),
    ax[0].set_xlabel("Diameter ($\mu $m)", fontsize=18)
    ax[0].xaxis.set_major_locator(plt.MaxNLocator(6))
    ax[0].set_ylabel("Volume Density (%)", fontsize=18)
    ax[0].tick_params(axis='both', which='major', labelsize=16)
    leg_00 = ax[0].legend(fontsize=16, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')
    plt.tight_layout() 
    fig.subplots_adjust(top=0.9)
    ax[0].set_xlim(0,max_x_value)
    for tick in ax[0].get_xticklabels():
        tick.set_fontname("Arial")
    for tick in ax[0].get_yticklabels():
        tick.set_fontname("Arial")
    ax[0].xaxis.label.set_color('black')
    ax[0].yaxis.label.set_color('black')
    ax[0].tick_params(axis='both', which='major', colors='black')
    for item in ([ax[0].xaxis.label, ax[0].yaxis.label] +
             ax[0].get_xticklabels() + ax[0].get_yticklabels()):
        item.set_fontsize(axfont)
    ax[0].set_ylim(bottom=0)
    ax[0].set_xlim(left=0)

    
    
    for spine in ['left','right','top','bottom']:
        ax[1].spines[spine].set_color('k')
    ax[1].set_facecolor('white')
    ax[1].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
            linewidth=2, color='k',label='Data' )
    ax[1].plot(x_vals_plot, lognormal_vals_curve_initial,
            linewidth=2, color='b',label='Lognormal' )
    ax[1].set_title('Initial parameters for Log-normal ', fontsize=16, fontname="Arial", weight="bold")
    ax[1].plot(x_vals_plot, lognormal_vals_curve_initial,
            linewidth=2, color='w', alpha=0, label=('A = %.2f\nmu = %.2f\nsigma = %.2f ' % (initial_params_lognormal_[0],initial_params_lognormal_[1],initial_params_lognormal_[2])  ))  
    ax[1].set_xlabel("Diameter ($\mu $m)", fontsize=18)
    ax[1].xaxis.set_major_locator(plt.MaxNLocator(6))
    ax[1].set_ylabel("Volume Density (%)", fontsize=18)
    ax[1].tick_params(axis='both', which='major', labelsize=16)
    leg_00 = ax[1].legend(fontsize=16, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')
    plt.tight_layout()
    ax[1].set_xlim(0,max_x_value)
    for tick in ax[1].get_xticklabels():
        tick.set_fontname("Arial")
    for tick in ax[1].get_yticklabels():
        tick.set_fontname("Arial")
    ax[1].xaxis.label.set_color('black')
    ax[1].yaxis.label.set_color('black')
    ax[1].tick_params(axis='both', which='major', colors='black')
    for item in ([ax[1].xaxis.label, ax[1].yaxis.label] +
             ax[1].get_xticklabels() + ax[1].get_yticklabels()):
        item.set_fontsize(axfont)
    ax[1].set_ylim(bottom=0)
    ax[1].set_xlim(left=0)
    
        
    #Saving plots in a folder called output
    plot_name = (name.strip('\n')+"_output.pdf")
    fig.savefig('output/Graphs_of_initial_parameters/'+name+"initial_parameters.pdf", bbox_inches = 'tight', dpi=300)


In [ ]:
#Getting values for all graphs to plot

#x values
x_vals_plot = np.arange(0.5,max_x_value,0.1)

    
#Normal - normal
normal_vals_curve = [normal_func(i_, *output_normal_fit_params_list) 
                         for i_ in x_vals_plot]



#lognormal-lognormal
lognormal_vals_curve = [lognormal_func(i_, *output_lognormal_fit_params_list) 
                         for i_ in x_vals_plot]





In [ ]:
# #Plotting everything with the scale the fitting has been performed against
# # Note - the if-else loops are used so that if the fitting failed then the graph is blank and says fitting failed
# # Note - saves the pdf in a folder called output

# #setting up plots
# fig, ax = plt.subplots(3,1,figsize=(15,25))
# fig.suptitle(name, fontsize=20, y=1.01,   fontname="Arial", weight="bold")
# fig.subplots_adjust(top=0.95)
# fig.tight_layout(h_pad=8)
# font = font_manager.FontProperties(family='Arial',
#                                    style='normal', size=11)
# axfont = 11


# #normal graph
# for spine in ['left','right','top','bottom']:
#     ax[0].spines[spine].set_color('k')
# ax[0].set_facecolor('white')
# if fitting_failed_normal == "yes":
#     ax[0].text(0.4, 0.5, 'Normal fitting failed', fontsize = 16)
# else:
#     ax[0].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
#             linewidth=2, color='lightgrey',label='Data' )
#     ax[0].plot(x_vals_plot, normal_vals_curve,
#             linewidth=2, color='#009e73ff',label='Normal'  )
#     ax[0].plot(x_vals_plot, normal_vals_curve,
#                linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
#            (normal_uncertainity, normal_standard_error_of_regression))
#     leg_00 = ax[0].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
#     leg_00.get_frame().set_edgecolor('k')

# ax[0].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
# ax[0].xaxis.set_major_locator(plt.MaxNLocator(6))
# ax[0].set_ylabel("Volume Density (%)", fontsize=11, fontname="Arial")
# ax[0].tick_params(axis='both', which='major', labelsize=11)

# for item in ([ax[0].xaxis.label, ax[0].yaxis.label] +
#              ax[0].get_xticklabels() + ax[0].get_yticklabels()):
#     item.set_fontsize(axfont)

# ax[0].set_title('Normal', fontsize=16, fontname="Arial", weight="bold")

# for tick in ax[0].get_xticklabels():
#     tick.set_fontname("Arial")
# for tick in ax[0].get_yticklabels():
#     tick.set_fontname("Arial")
# ax[0].xaxis.label.set_color('black')
# ax[0].yaxis.label.set_color('black')
# ax[0].tick_params(axis='both', which='major', colors='black')
# ax[0].set_ylim(bottom=0)
# ax[0].set_xlim(left=0)
# ax[0].set_xlim(0,max_x_value)
# ax[0].spines['top'].set_color('white')
# ax[0].spines['right'].set_color('white')

    
# #lognormal-lognormal graph
# for spine in ['left','right','top','bottom']:
#     ax[1].spines[spine].set_color('k')
#     ax[1].set_facecolor('white')
# if fitting_failed_lognormal == "yes":
#     ax[1].text(0.4, 0.5, 'Lognormal fitting failed', fontsize = 16)
# else:
#     ax[1].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
#             linewidth=2, color='lightgrey',label='Data' )
#     ax[1].plot(x_vals_plot, lognormal_vals_curve,
#             linewidth=2, color='#009e73ff',label='Log-normal - Log-normal'  )
#     ax[1].plot(x_vals_plot, lognormal_vals_curve,
#                linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
#            (lognormal_uncertainity, lognormal_standard_error_of_regression))
#     leg_00 = ax[1].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
#     leg_00.get_frame().set_edgecolor('k')

# ax[1].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
# ax[1].xaxis.set_major_locator(plt.MaxNLocator(6))
# ax[1].set_ylabel("Volume Density (%)", fontsize=11, fontname="Arial")
# ax[1].tick_params(axis='both', which='major', labelsize=11)

# for item in ([ax[1].xaxis.label, ax[1].yaxis.label] +
#              ax[1].get_xticklabels() + ax[1].get_yticklabels()):
#     item.set_fontsize(axfont)

# ax[1].set_title('Log-normal', fontsize=16, fontname="Arial", weight="bold")

# for tick in ax[1].get_xticklabels():
#     tick.set_fontname("Arial")
# for tick in ax[1].get_yticklabels():
#     tick.set_fontname("Arial")
# ax[1].xaxis.label.set_color('black')
# ax[1].yaxis.label.set_color('black')
# ax[1].tick_params(axis='both', which='major', colors='black')
# ax[1].set_ylim(bottom=0)
# ax[1].set_xlim(left=0)
# ax[1].set_xlim(0,max_x_value)
# ax[1].spines['top'].set_color('white')
# ax[1].spines['right'].set_color('white')

    

# #comparing all graphs
# for spine in ['left','right','top','bottom']:
#     ax[2].spines[spine].set_color('k')
# ax[2].set_facecolor('white')
# ax[2].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_density'].tolist(),
#             linewidth=2, color='lightgrey',label='Data' )
# if fitting_failed_normal == "yes":
#     pass
# else:
#     ax[2].plot(x_vals_plot, normal_vals_curve, 
#         linewidth=2, color='#1e88e5ff',label='N-N, S =%.5f\n' % (normal_standard_error_of_regression))  
# if fitting_failed_lognormal == "yes":
#     pass
# else:
#     ax[2].plot(x_vals_plot, lognormal_vals_curve,
#         linewidth=2, color='#ffc107ff',label='L-L, S =%.5f\n' % (lognormal_standard_error_of_regression))
# leg_00 = ax[2].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
# leg_00.get_frame().set_edgecolor('k')

# ax[2].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
# ax[2].xaxis.set_major_locator(plt.MaxNLocator(6))
# ax[2].set_ylabel("Volume Density (%)", fontsize=11, fontname="Arial")
# ax[2].tick_params(axis='both', which='major', labelsize=11)

# for item in ([ax[2].xaxis.label, ax[2].yaxis.label] +
#              ax[2].get_xticklabels() + ax[2].get_yticklabels()):
#     item.set_fontsize(axfont)

# ax[2].set_title('All fits', fontsize=16, fontname="Arial", weight="bold")

# for tick in ax[2].get_xticklabels():
#     tick.set_fontname("Arial")
# for tick in ax[2].get_yticklabels():
#     tick.set_fontname("Arial")
# ax[2].xaxis.label.set_color('black')
# ax[2].yaxis.label.set_color('black')
# ax[2].tick_params(axis='both', which='major', colors='black')
# ax[2].set_ylim(bottom=0)
# ax[2].set_xlim(left=0)
# ax[2].set_xlim(0,max_x_value)
# ax[2].spines['top'].set_color('white')
# ax[2].spines['right'].set_color('white')


# #Saving plots in a folder called output
# plot_name = (name.strip('\n')+"_output.pdf")
# fig.savefig('output/PDFs/'+name+".pdf", bbox_inches = 'tight', dpi=300)

In [ ]:
#Plotting everything with y adjusted scale
# Combining all plots together in one pdf file
# Note - the if-else loops are used so that if the fitting failed then the graph is blank and says fitting failed
# Note - saves the pdf in a folder called output

#Sums the Diff_volume_density column and convert to %
summed = input_data_df['Diff_volume_density'].sum()
input_data_df['Diff_volume_percentage'] = 100*input_data_df['Diff_volume_density']/summed
diff_volume_percentage_input_list = input_data_df['Diff_volume_percentage'].tolist()
conversion_factor = 100/summed
    
#Normal - normal
normal_vals_curve_adjusted_y = [i_ * conversion_factor for i_ in normal_vals_curve]

#lognormal-lognormal
lognormal_vals_curve_adjusted_y = [i_ * conversion_factor for i_ in lognormal_vals_curve]


#Adjusting values to plot by dividing by summed and multiplying by 100% (or just * by conversion_factor)

#x values
x_vals_plot = np.arange(0.5,max_x_value,0.1)



#setting up plots
fig, ax = plt.subplots(3,1,figsize=(15,25))
fig.suptitle(name, fontsize=20, y=1.01,   fontname="Arial", weight="bold")
fig.subplots_adjust(top=0.95)
fig.tight_layout(h_pad=8)
font = font_manager.FontProperties(family='Arial',
                                   style='normal', size=11)
axfont = 11


#normal graph
for spine in ['left','right','top','bottom']:
    ax[0].spines[spine].set_color('k')
ax[0].set_facecolor('white')
if fitting_failed_normal == "yes":
    ax[0].text(0.4, 0.5, 'Normal fitting failed', fontsize = 16)
else:
    ax[0].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_percentage'].tolist(),
            linewidth=2, color='lightgrey',label='Data' )
    ax[0].plot(x_vals_plot, normal_vals_curve_adjusted_y,
            linewidth=2, color='#009e73ff',label='Normal'  )
    ax[0].plot(x_vals_plot, normal_vals_curve_adjusted_y,
               linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
           (normal_uncertainity, normal_standard_error_of_regression))
    leg_00 = ax[0].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')

ax[0].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
ax[0].xaxis.set_major_locator(plt.MaxNLocator(6))
ax[0].set_ylabel("Relative Volume (%)", fontsize=11, fontname="Arial")
ax[0].tick_params(axis='both', which='major', labelsize=11)

for item in ([ax[0].xaxis.label, ax[0].yaxis.label] +
             ax[0].get_xticklabels() + ax[0].get_yticklabels()):
    item.set_fontsize(axfont)

ax[0].set_title('Normal', fontsize=16, fontname="Arial", weight="bold")

for tick in ax[0].get_xticklabels():
    tick.set_fontname("Arial")
for tick in ax[0].get_yticklabels():
    tick.set_fontname("Arial")
ax[0].xaxis.label.set_color('black')
ax[0].yaxis.label.set_color('black')
ax[0].tick_params(axis='both', which='major', colors='black')
ax[0].set_ylim(bottom=0)
ax[0].set_xlim(left=0)
ax[0].set_xlim(0,max_x_value)
ax[0].spines['top'].set_color('white')
ax[0].spines['right'].set_color('white')

    
#lognormal-lognormal graph
for spine in ['left','right','top','bottom']:
    ax[1].spines[spine].set_color('k')
    ax[1].set_facecolor('white')
if fitting_failed_lognormal == "yes":
    ax[1].text(0.4, 0.5, 'Lognormal fitting failed', fontsize = 16)
else:
    ax[1].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_percentage'].tolist(),
            linewidth=2, color='lightgrey',label='Data' )
    ax[1].plot(x_vals_plot, lognormal_vals_curve_adjusted_y,
            linewidth=2, color='#009e73ff',label='Log-normal - Log-normal'  )
    ax[1].plot(x_vals_plot, lognormal_vals_curve_adjusted_y,
               linewidth=2, color='w', alpha=0, label='Total uncertainty = %.2f\nStandard error of regression =%.5f' %
           (lognormal_uncertainity, lognormal_standard_error_of_regression))
    leg_00 = ax[1].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
    leg_00.get_frame().set_edgecolor('k')

ax[1].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
ax[1].xaxis.set_major_locator(plt.MaxNLocator(6))
ax[1].set_ylabel("Relative Volume (%)", fontsize=11, fontname="Arial")
ax[1].tick_params(axis='both', which='major', labelsize=11)

for item in ([ax[1].xaxis.label, ax[1].yaxis.label] +
             ax[1].get_xticklabels() + ax[1].get_yticklabels()):
    item.set_fontsize(axfont)

ax[1].set_title('Log-normal', fontsize=16, fontname="Arial", weight="bold")

for tick in ax[1].get_xticklabels():
    tick.set_fontname("Arial")
for tick in ax[1].get_yticklabels():
    tick.set_fontname("Arial")
ax[1].xaxis.label.set_color('black')
ax[1].yaxis.label.set_color('black')
ax[1].tick_params(axis='both', which='major', colors='black')
ax[1].set_ylim(bottom=0)
ax[1].set_xlim(left=0)
ax[1].set_xlim(0,max_x_value)
ax[1].spines['top'].set_color('white')
ax[1].spines['right'].set_color('white')

    

#comparing all graphs
for spine in ['left','right','top','bottom']:
    ax[2].spines[spine].set_color('k')
ax[2].set_facecolor('white')
ax[2].plot(input_data_df['Bin_midpoint'].tolist(), input_data_df['Diff_volume_percentage'].tolist(),
            linewidth=2, color='lightgrey',label='Data' )
if fitting_failed_normal == "yes":
    pass
else:
    ax[2].plot(x_vals_plot, normal_vals_curve_adjusted_y, 
        linewidth=2, color='#1e88e5ff',label='N, S =%.5f\n' % (normal_standard_error_of_regression))  
if fitting_failed_lognormal == "yes":
    pass
else:
    ax[2].plot(x_vals_plot, lognormal_vals_curve_adjusted_y,
        linewidth=2, color='#ffc107ff',label='L, S =%.5f\n' % (lognormal_standard_error_of_regression))
leg_00 = ax[2].legend(fontsize=11, loc='upper right', ncol=1,facecolor='white', framealpha=1, prop=font)
leg_00.get_frame().set_edgecolor('k')

ax[2].set_xlabel("Diameter ($\mu $m)", fontsize=11, fontname ="Arial")
ax[2].xaxis.set_major_locator(plt.MaxNLocator(6))
ax[2].set_ylabel("Relative Volume (%)", fontsize=11, fontname="Arial")
ax[2].tick_params(axis='both', which='major', labelsize=11)

for item in ([ax[2].xaxis.label, ax[2].yaxis.label] +
             ax[2].get_xticklabels() + ax[2].get_yticklabels()):
    item.set_fontsize(axfont)

ax[2].set_title('All fits', fontsize=16, fontname="Arial", weight="bold")

for tick in ax[2].get_xticklabels():
    tick.set_fontname("Arial")
for tick in ax[2].get_yticklabels():
    tick.set_fontname("Arial")
ax[2].xaxis.label.set_color('black')
ax[2].yaxis.label.set_color('black')
ax[2].tick_params(axis='both', which='major', colors='black')
ax[2].set_ylim(bottom=0)
ax[2].set_xlim(left=0)
ax[2].set_xlim(0,max_x_value)
ax[2].spines['top'].set_color('white')
ax[2].spines['right'].set_color('white')


#Saving plots in a folder called output
plot_name = (name.strip('\n')+"_output.pdf")
fig.savefig('output/PDFs/'+name+".pdf", bbox_inches = 'tight', dpi=300)

In [ ]:
#Getting the fitting parameters from the different fits
normal_fitting_parameters = (A_normal, A_normal_uncertainty, mu_normal, mu_normal_uncertainty, sigma_normal, sigma_normal_uncertainty)
lognormal_fitting_parameters = (A_lognormal, A_lognormal_uncertainty, mu_lognormal, mu_lognormal_uncertainty, sigma_lognormal, sigma_lognormal_uncertainty)
#Joining together the sample name and fitting parameters
name_str = (name,) #converts the sample name into a tuple so it is in the same format as the parameters
fitting_parameters = (name_str + normal_fitting_parameters + lognormal_fitting_parameters)
    

#Getting the starch parameters from the different fits
normal_parameters = (granule_diameter_normal, normal_variance, normal_uncertainity, normal_standard_error_of_regression)
lognormal_parameters = (granule_diameter_lognormal, lognormal_variance, lognormal_uncertainity, lognormal_standard_error_of_regression)
#Joining together the sample name and fitting parameters
starch_fitting_parameters = (name_str + normal_parameters + lognormal_parameters)